In [ ]:
import glob, json, time, platform, subprocess, sys, importlib.util
if importlib.util.find_spec('sentence_transformers') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
assert torch.cuda.is_available(), 'Kaggle GPU is required'
f = lambda name: glob.glob('/kaggle/input/**/' + name, recursive=True)[0]
data = json.load(open(f('queries.json')))
z = np.load(f('curated_doc_embeddings.npz'))
D = z['D'].astype(np.float32)
starts = z['lab_starts'].astype(int)
lab_ids = z['lab_ids'].tolist()
t0 = time.time()
model = SentenceTransformer('BAAI/bge-m3', device='cuda').half()
load_s = time.time() - t0
t0 = time.time()
E = model.encode(data['phrases'], batch_size=32, normalize_embeddings=True, show_progress_bar=False)
encode_s = time.time() - t0
scores = np.maximum.reduceat(np.asarray(E, dtype=np.float32) @ D.T, starts, axis=1)
np.savez_compressed('/kaggle/working/query_scores.npz', scores=scores, lab_ids=np.asarray(lab_ids))
json.dump({'gpu':torch.cuda.get_device_name(0), 'torch':torch.__version__, 'model':'BAAI/bge-m3', 'model_load_s':load_s, 'encode_s':encode_s, 'phrases':len(data['phrases']), 'documents':len(D), 'labs':len(lab_ids)}, open('/kaggle/working/run_metadata.json','w'), indent=2)
print('GPU', torch.cuda.get_device_name(0), 'phrases',len(data['phrases']), 'encode_s',encode_s)
